In [ ]:
# ================== CELL 1: CONFIGURATION ==================
import os
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "image_denoising":
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
elif NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = "gemini-3.5-flash"

INPUT_IMAGE_PATH = PROJECT_ROOT / "input_images" / "noise_image" / "test_image_3.png"
OUTPUT_IMAGE_PATH = PROJECT_ROOT / "outputs" / "image_denoising" / "clean_front_face_output.png"

OUTPUT_IMAGE_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Setup complete. Model: {GEMINI_MODEL}")
print(f"API key loaded: {'yes' if API_KEY else 'no — add GEMINI_API_KEY to .env'}")

In [ ]:
# ================== CELL 2: INITIALIZATION ==================
import io
import json
import re
import cv2
import numpy as np
from google import genai
from google.genai import types
from google.genai.errors import ClientError
from PIL import Image
import matplotlib.pyplot as plt

if not API_KEY:
    raise ValueError(
        "GEMINI_API_KEY is missing. Copy .env.example to .env and set your key."
    )

client = genai.Client(api_key=API_KEY)

# Strictly forcing the model to output exactly 4 corner points in a standard JSON schema
CORNER_JSON_SCHEMA = {
    "type": "array",
    "description": "Four corner points: top-left, top-right, bottom-right, bottom-left",
    "items": {
        "type": "array",
        "items": {"type": "number"},
        "minItems": 2,
        "maxItems": 2,
    },
    "minItems": 4,
    "maxItems": 4,
}

CORNER_PROMPT = """
Analyze this construction rebar cage image.
Return the coordinates [y, x] of the 4 outer corners where the closest, outermost front-facing rectangular grid boundary meets.
Corners are where the leftmost vertical bar, rightmost vertical bar, topmost horizontal tie, and bottommost horizontal tie intersect at the edges.

Ignore side faces, deep back-layer bars, cranes, and sky. 
Coordinates must be scaled on a 0-1000 normalized range relative to the image dimensions.
Order must be exactly: [top-left, top-right, bottom-right, bottom-left]. Return ONLY raw JSON.
"""
print("🚀 Client initialized successfully.")

In [ ]:
# ================== CELL 3: ADVANCED MASKING ENGINE ==================
import json
import re
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
from google.genai import types

MASK_GENERATION_PROMPT = """
Analyze the provided construction rebar column image carefully. 
Your task is to identify and isolate ONLY the absolute front-facing grid layer (the 5 vertical bars and 9 horizontal bars closest to the camera) and the 4 corner concrete spacer blocks.

Generate a valid JSON object containing:
1. "vertical_bars": An array of centerlines representing the 5 vertical front-facing bars. Each bar is represented by its start and end point center coordinates.
2. "horizontal_bars": An array of centerlines representing the 9 horizontal front-facing ties. Each tie is represented by its start and end point center coordinates.
3. "spacer_blocks": An array of bounding boxes tracing the 4 corner concrete spacer blocks.

Do not include any background elements: erase or omit the bars inside the cage, the background layers, the cranes, the scaffolding, the ground, and the sky.
Specifically, exclude all diagonal cross-ties, deep interior rebar cages, and depth-defining columns on the left and right sides.

Return your response strictly in this JSON format:
{
  "vertical_bars": [
    {"x1": int, "y1": int, "x2": int, "y2": int}
  ],
  "horizontal_bars": [
    {"x1": int, "y1": int, "x2": int, "y2": int}
  ],
  "spacer_blocks": [
    {"ymin": int, "xmin": int, "ymax": int, "xmax": int}
  ]
}
Note: Scale all coordinate values on a normalized 0 to 1000 system relative to the image bounds. Return ONLY the raw JSON block.
"""

# Define a strict JSON schema to guarantee centerline and bounding box formatting on the API side
REBAR_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "vertical_bars": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "x1": {"type": "INTEGER"},
                    "y1": {"type": "INTEGER"},
                    "x2": {"type": "INTEGER"},
                    "y2": {"type": "INTEGER"}
                },
                "required": ["x1", "y1", "x2", "y2"]
            }
        },
        "horizontal_bars": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "x1": {"type": "INTEGER"},
                    "y1": {"type": "INTEGER"},
                    "x2": {"type": "INTEGER"},
                    "y2": {"type": "INTEGER"}
                },
                "required": ["x1", "y1", "x2", "y2"]
            }
        },
        "spacer_blocks": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "ymin": {"type": "INTEGER"},
                    "xmin": {"type": "INTEGER"},
                    "ymax": {"type": "INTEGER"},
                    "xmax": {"type": "INTEGER"}
                },
                "required": ["ymin", "xmin", "ymax", "xmax"]
            }
        }
    },
    "required": ["vertical_bars", "horizontal_bars", "spacer_blocks"]
}

def safe_extract_box(box_item) -> tuple[int, int, int, int]:
    """
    Robust box coordinate parser that extracts ymin, xmin, ymax, xmax case-insensitively,
    supporting underscores, hyphens, alternative names, and lists to prevent KeyError.
    """
    if isinstance(box_item, (list, tuple)) and len(box_item) >= 4:
        return int(box_item[0]), int(box_item[1]), int(box_item[2]), int(box_item[3])
    
    if isinstance(box_item, dict):
        clean_dict = {str(k).lower().replace("_", "").replace("-", ""): v for k, v in box_item.items()}
        ymin = clean_dict.get("ymin", clean_dict.get("y1", clean_dict.get("top", 0)))
        xmin = clean_dict.get("xmin", clean_dict.get("x1", clean_dict.get("left", 0)))
        ymax = clean_dict.get("ymax", clean_dict.get("y2", clean_dict.get("bottom", 1000)))
        xmax = clean_dict.get("xmax", clean_dict.get("x2", clean_dict.get("right", 1000)))
        return int(ymin), int(xmin), int(ymax), int(xmax)
        
    return 0, 0, 1000, 1000

def safe_extract_line(line_item) -> tuple[int, int, int, int]:
    """Robustly extracts line coordinates (x1, y1, x2, y2) supporting case variations."""
    if isinstance(line_item, (list, tuple)) and len(line_item) >= 4:
        return int(line_item[0]), int(line_item[1]), int(line_item[2]), int(line_item[3])
    
    if isinstance(line_item, dict):
        clean_dict = {str(k).lower().replace("_", "").replace("-", ""): v for k, v in line_item.items()}
        x1 = clean_dict.get("x1", clean_dict.get("startx", 0))
        y1 = clean_dict.get("y1", clean_dict.get("starty", 0))
        x2 = clean_dict.get("x2", clean_dict.get("endx", 1000))
        y2 = clean_dict.get("y2", clean_dict.get("endy", 1000))
        return int(x1), int(y1), int(x2), int(y2)
        
    return 0, 0, 1000, 1000

def draw_perspective_line(mask, pt1, pt2, max_dim, val, bottom_ratio, top_ratio):
    """
    Draws a line with dynamic thickness that tapers from bottom to top to match perspective projection.
    This prevents wide masks at the top of the rebar cage where background elements are tightly clustered.
    """
    x1, y1 = pt1
    x2, y2 = pt2
    h_img = mask.shape[0]
    
    length = np.hypot(x2 - x1, y2 - y1)
    if length < 1e-5:
        return
        
    # Scale steps to avoid segment gaps on very long lines
    num_steps = int(max(15, min(150, length / 8.0)))
    
    for i in range(num_steps):
        t = i / float(num_steps)
        t_next = (i + 1) / float(num_steps)
        
        curr_x = int(x1 + t * (x2 - x1))
        curr_y = int(y1 + t * (y2 - y1))
        next_x = int(x1 + t_next * (x2 - x1))
        next_y = int(y1 + t_next * (y2 - y1))
        
        # Normalize y coordinate (0.0 at top, 1.0 at bottom)
        y_norm = curr_y / float(h_img)
        y_norm = max(0.0, min(1.0, y_norm))
        
        # Calculate localized perspective thickness
        ratio = top_ratio + (bottom_ratio - top_ratio) * y_norm
        thickness = max(2, int(max_dim * ratio))
        
        cv2.line(mask, (curr_x, curr_y), (next_x, next_y), val, thickness)

def clean_rebar_structure(image_path: Path, output_path: Path):
    """Leverages Gemini 3.5 Flash context mapping to programmatically erase depth layers without cropping."""
    if not image_path.exists():
        print(f"❌ Error: Input image missing at {image_path}")
        return

    print("1. Sending image data to Gemini 3.5 Flash for pixel-layer analysis...")
    pil_img = Image.open(image_path).convert("RGB")
    
    # Configure strict JSON enforcement configuration
    config = types.GenerateContentConfig.model_validate(
        {
            "response_mime_type": "application/json",
            "response_schema": REBAR_SCHEMA,
            "temperature": 0.1,
        }
    )
    
    text_data = ""
    try:
        # Utilizing the stable gemini-3.5-flash text/multimodal generation pipeline
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=[MASK_GENERATION_PROMPT, pil_img],
            config=config
        )
        
        text_data = response.text or ""
        
        print("2. Parsing and cleaning JSON structure response...")
        json_match = re.search(r"\{[\s\S]*\}", text_data)
        
        if json_match:
            clean_json_str = json_match.group(0)
            mask_data = json.loads(clean_json_str)
        else:
            mask_data = json.loads(text_data.strip())
        
        print("3. Centerline skeleton verified! Initializing perspective-tapered masks...")
        src_img = cv2.imread(str(image_path))
        if src_img is None:
            raise ValueError(f"Could not read image: {image_path}")
        h, w, c = src_img.shape
        max_dim = max(h, w)
        
        # Initialize entire canvas as Definite Background (0) to isolate front-plane regions strictly
        gc_mask = np.zeros((h, w), dtype=np.uint8) # 0 = cv2.GC_BGD

        # Compile centerline lists
        target_lines = mask_data.get("vertical_bars", []) + mask_data.get("horizontal_bars", [])

        # Step A: Draw wide perspective-tapered corridors to define GrabCut search windows (Probable Background)
        # 2 = cv2.GC_PR_BGD. Ratios taper from 4.8% at the bottom to 1.5% at the top.
        for bar in target_lines:
            x1, y1, x2, y2 = safe_extract_line(bar)
            pt1 = (int(x1 / 1000.0 * w), int(y1 / 1000.0 * h))
            pt2 = (int(x2 / 1000.0 * w), int(y2 / 1000.0 * h))
            draw_perspective_line(gc_mask, pt1, pt2, max_dim, cv2.GC_PR_BGD, 0.048, 0.015)

        # Step B: Draw perspective-tapered probable rebar foreground cylinders
        # 3 = cv2.GC_PR_FGD. Ratios taper from 2.8% at the bottom to 0.9% at the top.
        for bar in target_lines:
            x1, y1, x2, y2 = safe_extract_line(bar)
            pt1 = (int(x1 / 1000.0 * w), int(y1 / 1000.0 * h))
            pt2 = (int(x2 / 1000.0 * w), int(y2 / 1000.0 * h))
            draw_perspective_line(gc_mask, pt1, pt2, max_dim, cv2.GC_PR_FGD, 0.028, 0.009)

        # Step C: Draw ultra-tight Definite Foreground core seeds strictly down centerlines
        # 1 = cv2.GC_FGD. Ratios taper from 1.2% at the bottom to 0.4% at the top.
        for bar in target_lines:
            x1, y1, x2, y2 = safe_extract_line(bar)
            pt1 = (int(x1 / 1000.0 * w), int(y1 / 1000.0 * h))
            pt2 = (int(x2 / 1000.0 * w), int(y2 / 1000.0 * h))
            draw_perspective_line(gc_mask, pt1, pt2, max_dim, cv2.GC_FGD, 0.012, 0.004)
            
        # Draw concrete spacer block masks securely as Definite Foreground (1)
        for block in mask_data.get("spacer_blocks", []):
            ymin, xmin, ymax, xmax = safe_extract_box(block)
            p_ymin = int((ymin / 1000.0) * h)
            p_xmin = int((xmin / 1000.0) * w)
            p_ymax = int((ymax / 1000.0) * h)
            p_xmax = int((xmax / 1000.0) * w)
            cv2.rectangle(gc_mask, (p_xmin, p_ymin), (p_xmax, p_ymax), cv2.GC_FGD, -1)
            
        # Step D: Force Sky Highlights (>210) & Deep Interior Cage Shadows (<40) inside the active search corridor
        # back to Definite Background (0). This prevents GrabCut from bleeding into deep dark or bright spaces.
        gray = cv2.cvtColor(src_img, cv2.COLOR_BGR2GRAY)
        sky_pixels = (gray > 210)
        shadow_pixels = (gray < 40)
        prune_mask = sky_pixels | shadow_pixels
        
        # Enforce pruning strictly on non-seed core pixels
        gc_mask[(prune_mask) & (gc_mask != cv2.GC_FGD)] = cv2.GC_BGD

        print("4. Executing GrabCut segmentation modeling (5 iterative sweeps)...")
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        grabcut_rect = (0, 0, w, h)  # ignored in GC_INIT_WITH_MASK; required by OpenCV stubs
        
        cv2.grabCut(src_img, gc_mask, grabcut_rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_MASK)
        
        # Build binary mask based on finalized definite + probable foreground zones
        final_mask = np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 255, 0).astype('uint8')
        
        # Post-process: Close holes and smooth structural borders
        kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, kernel_open)
        final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_CLOSE, kernel_close)
        
        # Step E: Filter out disconnected background artifacts and floating noise using Connected Components
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(final_mask)
        min_area = int(h * w * 0.0015) # Filter out components smaller than 0.15% of total image area
        
        cleaned_mask = np.zeros_like(final_mask)
        for i in range(1, num_labels): # Skip background index 0
            if stats[i, cv2.CC_STAT_AREA] >= min_area:
                cleaned_mask[labels == i] = 255
                
        # Final edge smoothing
        cleaned_mask = cv2.GaussianBlur(cleaned_mask, (3, 3), 0)
        
        print("5. Rendering textures smoothly onto clean solid gray backdrop...")
        studio_canvas = np.full_like(src_img, 235)  # Clean light studio backdrop (#EAEAEA)
        mask_3ch = cv2.merge([cleaned_mask, cleaned_mask, cleaned_mask])
        
        # Overlay original high-quality pixels only on the mask
        np.copyto(studio_canvas, src_img, where=(mask_3ch > 127))
        
        # Save output
        cv2.imwrite(str(output_path), studio_canvas)
        print(f"✨ SUCCESS! Perfect photographic rebar front grid saved to: {output_path}")
        
    except json.JSONDecodeError as je:
        print(f"❌ JSON Parsing failed. The model returned invalid formatting.")
        print(f"Raw response preview was:\n{text_data[:500]}")
    except Exception as e:
        print(f"❌ Masking pipeline execution failed: {e}")

In [ ]:
# ================== CELL 4: PIPELINE RUNNER ==================
clean_rebar_structure(INPUT_IMAGE_PATH, OUTPUT_IMAGE_PATH)

In [ ]:
# ================== CELL 5: SIDE-BY-SIDE PLOT ==================
if INPUT_IMAGE_PATH.exists() and OUTPUT_IMAGE_PATH.exists():
    img_in_bgr = cv2.imread(str(INPUT_IMAGE_PATH))
    img_out_bgr = cv2.imread(str(OUTPUT_IMAGE_PATH))
    if img_in_bgr is None or img_out_bgr is None:
        raise FileNotFoundError("Could not load one or both images for display.")

    img_in = cv2.cvtColor(img_in_bgr, cv2.COLOR_BGR2RGB)
    img_out = cv2.cvtColor(img_out_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(img_in)
    axes[0].set_title("Original Structure (With Background Noise)")
    axes[0].axis("off")
    
    axes[1].imshow(img_out)
    axes[1].set_title("Clean Isolated Front Face Result")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()